# Rule Based Chunking

In [1]:
import re
import json
import os
from dataclasses import dataclass, asdict
from typing import List, Optional
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
EXTRACTED_TEXT_PATH = "../data/extracted_text/granite_docling_output.txt"
OUTPUT_PATH         = "../data/chunks/rulebased_chunked_dataset.jsonl"
STATS_PATH          = "../data/chunks/rulebased_chunking_stats.json"

# ── Chunking parameters ───────────────────────────────────────────────────────
TARGET_CHUNK_TOKENS  = 400   # ideal chunk size for retrieval
MAX_CHUNK_TOKENS     = 512   # hard ceiling — never exceed this
MIN_CHUNK_TOKENS     = 60    # below this a chunk can't generate a QA pair
OVERLAP_TOKENS       = 50    # carried over from previous chunk at boundaries

# ── Token estimation ──────────────────────────────────────────────────────────
# 3.2 chars/token — conservative for markdown with tables and sparse content
CHARS_PER_TOKEN      = 3.2

print("✓ Configuration loaded")
print(f"  Target chunk size : {TARGET_CHUNK_TOKENS} tokens")
print(f"  Max chunk size    : {MAX_CHUNK_TOKENS} tokens")
print(f"  Min chunk size    : {MIN_CHUNK_TOKENS} tokens")
print(f"  Overlap           : {OVERLAP_TOKENS} tokens")

✓ Configuration loaded
  Target chunk size : 400 tokens
  Max chunk size    : 512 tokens
  Min chunk size    : 60 tokens
  Overlap           : 50 tokens


In [2]:
#Chunk Dataclass
@dataclass
class Chunk:
    chunk_id:       str
    text:           str
    h1:             str        # top-level chapter
    h2:             str        # section
    h3:             str        # subsection
    breadcrumb:     str        # "Chapter > Section > Subsection"
    token_estimate: int
    char_count:     int
    chunk_type:     str        # "heading_section" | "procedure" | "warning" | "table" | "paragraph"

    def to_dict(self) -> dict:
        return asdict(self)


def make_chunk(
    chunk_id: int,
    text: str,
    h1: str,
    h2: str,
    h3: str,
    chunk_type: str = "paragraph",
) -> Chunk:
    breadcrumb = " > ".join(filter(None, [h1, h2, h3]))
    text = text.strip()
    return Chunk(
        chunk_id=f"chunk_{chunk_id:04d}",
        text=text,
        h1=h1,
        h2=h2,
        h3=h3,
        breadcrumb=breadcrumb,
        token_estimate=estimate_tokens(text),
        char_count=len(text),
        chunk_type=chunk_type,
    )


print("✓ Chunk dataclass defined")

✓ Chunk dataclass defined


In [3]:
#Token and Text Utilities
def estimate_tokens(text: str) -> int:
    """Conservative token estimate. No API call needed."""
    return int(len(text) / CHARS_PER_TOKEN)


def is_warning_block(text: str) -> bool:
    """Detect WARNING / CAUTION / NOTICE / NOTE blocks."""
    return bool(re.match(
        r"^\s*(WARNING|CAUTION|NOTICE|NOTE)\b",
        text, re.IGNORECASE
    ))


def is_procedure(text: str) -> bool:
    """Detect numbered step procedures."""
    return bool(re.search(
        r"^\s*(step\s+\d+|[1-9]\d*\.|\([1-9]\d*\))\s+\S",
        text, re.IGNORECASE | re.MULTILINE
    ))


def is_table(text: str) -> bool:
    """Detect markdown tables."""
    return bool(re.search(r"^\|.+\|", text, re.MULTILINE))


def detect_chunk_type(text: str) -> str:
    if is_table(text):
        return "table"
    if is_warning_block(text):
        return "warning"
    if is_procedure(text):
        return "procedure"
    if re.match(r"^#{1,3}\s+", text):
        return "heading_section"
    return "paragraph"


def clean_text(text: str) -> str:
    """
    Remove PDF extraction artifacts.
    Preserves all real manual content.
    """
    # Docling image placeholders
    text = re.sub(r"<!--\s*image\s*-->", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)

    # Lone page numbers (digit-only lines)
    text = re.sub(r"^\s*\d{1,3}\s*$", "", text, flags=re.MULTILINE)

    # Part numbers and copyright lines
    text = re.sub(r"^\s*Part No\.\s+\S+.*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*©.*$", "", text, flags=re.MULTILINE)

    # Collapse 3+ newlines → 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


print("✓ Utilities defined")

✓ Utilities defined


In [8]:
# ── Splitters ─────────────────────────────────────────────────────────────────

_HEADING_RE = re.compile(r"^(#{1,3})\s+(.+)$", re.MULTILINE)


def split_by_sentences(text: str, max_tokens: int, overlap_tokens: int) -> List[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks  = []
    current = ""
    overlap = ""

    for sentence in sentences:
        candidate = (overlap + " " + sentence).strip() if overlap else sentence

        if estimate_tokens(candidate) > max_tokens and current:
            chunks.append(current.strip())
            words = current.split()
            overlap_words = []
            overlap_len   = 0
            for w in reversed(words):
                overlap_len += len(w) / CHARS_PER_TOKEN
                if overlap_len > overlap_tokens:
                    break
                overlap_words.insert(0, w)
            overlap  = " ".join(overlap_words)
            current  = (overlap + " " + sentence).strip() if overlap else sentence
        else:
            current = candidate

    if current.strip():
        chunks.append(current.strip())

    return [c for c in chunks if c.strip()]


def split_by_paragraphs(
    text: str,
    max_tokens: int,
    overlap_tokens: int,
    heading_prefix: str = "",
) -> List[str]:
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks         = []
    current_paras  = []
    current_tokens = 0

    def flush(paras: List[str]) -> None:
        if not paras:
            return
        body = "\n\n".join(paras)
        full = f"{heading_prefix}\n\n{body}".strip() if heading_prefix else body
        chunks.append(full)

    for para in paragraphs:
        para_tokens = estimate_tokens(para)

        if para_tokens > max_tokens:
            flush(current_paras)
            current_paras  = []
            current_tokens = 0
            prefix = heading_prefix if not chunks else ""
            for sent_chunk in split_by_sentences(para, max_tokens, overlap_tokens):
                full = f"{prefix}\n\n{sent_chunk}".strip() if prefix else sent_chunk
                chunks.append(full)
            continue

        if current_tokens + para_tokens > max_tokens:
            flush(current_paras)
            if current_paras:
                overlap_para   = current_paras[-1]
                current_paras  = [overlap_para, para]
                current_tokens = estimate_tokens(overlap_para) + para_tokens
            else:
                current_paras  = [para]
                current_tokens = para_tokens
        else:
            current_paras.append(para)
            current_tokens += para_tokens

    flush(current_paras)
    return [c for c in chunks if c.strip()]


def split_table(text: str, max_tokens: int, heading_prefix: str = "") -> List[str]:
    lines = text.split("\n")
    header_lines = []
    data_lines   = []
    found_sep    = False

    for line in lines:
        if not found_sep and re.match(r"^\|[-| :]+\|", line):
            header_lines.append(line)
            found_sep = True
        elif not found_sep:
            header_lines.append(line)
        else:
            data_lines.append(line)

    header     = "\n".join(header_lines)
    header_tok = estimate_tokens(header)
    chunks     = []
    current    = []
    cur_tok    = header_tok

    for row in data_lines:
        row_tok = estimate_tokens(row)
        if cur_tok + row_tok > max_tokens and current:
            body = header + "\n" + "\n".join(current)
            full = f"{heading_prefix}\n\n{body}".strip() if heading_prefix else body
            chunks.append(full)
            current = [row]
            cur_tok = header_tok + row_tok
        else:
            current.append(row)
            cur_tok += row_tok

    if current:
        body = header + "\n" + "\n".join(current)
        full = f"{heading_prefix}\n\n{body}".strip() if heading_prefix else body
        chunks.append(full)

    return [c for c in chunks if c.strip()] or [text]


# ── Section Parser ────────────────────────────────────────────────────────────

def parse_heading_sections(text: str) -> List[dict]:
    matches = list(_HEADING_RE.finditer(text))

    if not matches:
        return [{
            "level": 1, "heading": "Document",
            "content": text, "h1": "Document", "h2": "", "h3": "",
            "heading_text": "Document"
        }]

    sections = []
    h1 = h2 = h3 = ""

    for i, match in enumerate(matches):
        level   = len(match.group(1))
        heading = match.group(2).strip()
        start   = match.end()
        end     = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()

        if level == 1:
            h1 = heading
            h2 = ""
            h3 = ""
        elif level == 2:
            h2 = heading
            h3 = ""
        elif level == 3:
            h3 = heading

        sections.append({
            "level":        level,
            "heading":      heading,
            "heading_text": "#" * level + " " + heading,
            "content":      content,
            "h1":           h1,
            "h2":           h2,
            "h3":           h3,
        })

    return sections


# ── Chunk Section ─────────────────────────────────────────────────────────────

def chunk_section(
    section:        dict,
    chunk_counter:  List[int],
    max_tokens:     int = TARGET_CHUNK_TOKENS,
    min_tokens:     int = MIN_CHUNK_TOKENS,
    overlap_tokens: int = OVERLAP_TOKENS,
) -> List[Chunk]:
    heading_prefix = section["heading_text"]
    content        = section["content"]
    h1, h2, h3     = section["h1"], section["h2"], section["h3"]
    chunks         = []

    def add(text: str, ctype: str = "auto") -> None:
        text = text.strip()
        if not text or estimate_tokens(text) < min_tokens:
            return
        if ctype == "auto":
            ctype = detect_chunk_type(text)
        chunk_counter[0] += 1
        chunks.append(make_chunk(chunk_counter[0], text, h1, h2, h3, ctype))

    if not content:
        add(heading_prefix, "heading_section")
        return chunks

    full_text    = f"{heading_prefix}\n\n{content}"
    full_tok     = estimate_tokens(full_text)
    content_type = detect_chunk_type(content)

    if full_tok <= max_tokens:
        add(full_text, content_type)
        return chunks

    if content_type == "table":
        for sub in split_table(content, max_tokens, heading_prefix):
            add(sub, "table")
        return chunks

    if content_type == "procedure":
        blocks = re.split(r"\n(?=(?:WARNING|CAUTION|NOTICE|NOTE)\b)", content)
        current_block = ""
        for block in blocks:
            candidate = (current_block + "\n\n" + block).strip() if current_block else block
            if estimate_tokens(candidate) > max_tokens and current_block:
                add(f"{heading_prefix}\n\n{current_block}".strip(), "procedure")
                current_block = block
            else:
                current_block = candidate
        if current_block:
            add(f"{heading_prefix}\n\n{current_block}".strip(), "procedure")
        return chunks

    for sub in split_by_paragraphs(content, max_tokens, overlap_tokens, heading_prefix):
        add(sub, content_type)

    return chunks


# ── Main Document Chunker ─────────────────────────────────────────────────────

def chunk_document(
    text:           str,
    max_tokens:     int = TARGET_CHUNK_TOKENS,
    min_tokens:     int = MIN_CHUNK_TOKENS,
    overlap_tokens: int = OVERLAP_TOKENS,
) -> List[Chunk]:
    print("Step 1/4 — Cleaning text...")
    text = clean_text(text)

    print("Step 2/4 — Parsing heading hierarchy...")
    sections = parse_heading_sections(text)
    print(f"  Found {len(sections)} sections")

    print("Step 3/4 — Chunking sections...")
    all_chunks    = []
    chunk_counter = [0]

    for sec in sections:
        new_chunks = chunk_section(
            section        = sec,
            chunk_counter  = chunk_counter,
            max_tokens     = max_tokens,
            min_tokens     = min_tokens,
            overlap_tokens = overlap_tokens,
        )
        all_chunks.extend(new_chunks)

    print(f"  Produced {len(all_chunks)} chunks")
    print("Step 4/4 — Done")
    return all_chunks


print("✓ All chunking functions defined")

✓ All chunking functions defined


In [9]:
#Save and Load
def save_chunks(chunks: List[Chunk], output_path: str) -> None:
    """Save chunks as JSONL — one JSON object per line."""
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk.to_dict(), ensure_ascii=False) + "\n")

    print(f"✓ Saved {len(chunks)} chunks → {output_path}")


def save_stats(stats: dict, stats_path: str) -> None:
    """Save stats as JSON for later inspection."""
    Path(stats_path).parent.mkdir(parents=True, exist_ok=True)
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved stats → {stats_path}")


def load_chunks(path: str) -> List[dict]:
    """Load chunks back from JSONL for inspection."""
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    return chunks


print("✓ Save/load functions defined")

✓ Save/load functions defined


In [10]:
#Run Pipeline
# ── Load extracted text ───────────────────────────────────────────────────────
print(f"Loading: {EXTRACTED_TEXT_PATH}")
with open(EXTRACTED_TEXT_PATH, "r", encoding="utf-8") as f:
    extracted_text = f.read()

print(f"  Characters : {len(extracted_text):,}")
print(f"  Est. tokens: {estimate_tokens(extracted_text):,}")

# ── Run chunker ───────────────────────────────────────────────────────────────
chunks = chunk_document(extracted_text)

# ── Validate ──────────────────────────────────────────────────────────────────
print("\nValidating chunks...")
chunks = validate_chunks(chunks)

# ── Stats ─────────────────────────────────────────────────────────────────────
stats = compute_stats(chunks)
print_stats(stats)

# ── Save ──────────────────────────────────────────────────────────────────────
save_chunks(chunks, OUTPUT_PATH)
save_stats(stats, STATS_PATH)

Loading: ../data/extracted_text/granite_docling_output.txt
  Characters : 1,945,221
  Est. tokens: 607,881
Step 1/4 — Cleaning text...
Step 2/4 — Parsing heading hierarchy...
  Found 4861 sections
Step 3/4 — Chunking sections...
  Produced 2737 chunks
Step 4/4 — Done

Validating chunks...
  [validate] Oversized chunk chunk_0038 (15107 tokens): '## Warning Buzzer\n\n- \\u00b7 *1: The master warning in the in'
  [validate] Oversized chunk chunk_0039 (15107 tokens): '## Warning Buzzer\n\n- \\u00b7 *1: The master warning in the in'
  [validate] Oversized chunk chunk_0425 (858 tokens): '## Automatic transmission models\n\n| Warning and indicator  m'
  [validate] Oversized chunk chunk_0875 (838 tokens): '## Automatic transmission models\n\n| Warning and indicator  m'
  [validate] Oversized chunk chunk_1367 (15107 tokens): '## Warning Buzzer\n\n- \\u00b7 *1: The master warning in the in'
  [validate] Oversized chunk chunk_1368 (15107 tokens): '## Warning Buzzer\n\n- \\u00b7 *1: The master war

In [11]:
#Inspect Sample
# Quick sanity check — print 5 random chunks to verify quality
import random

loaded = load_chunks(OUTPUT_PATH)
samples = random.sample(loaded, min(5, len(loaded)))

print(f"Total chunks in file: {len(loaded)}\n")
print("=" * 60)

for i, chunk in enumerate(samples):
    print(f"\nSample {i+1} — {chunk['chunk_id']} [{chunk['chunk_type']}]")
    print(f"Breadcrumb : {chunk['breadcrumb']}")
    print(f"Tokens     : {chunk['token_estimate']}")
    print(f"Characters : {chunk['char_count']}")
    print(f"Text preview:")
    print("-" * 40)
    print(chunk['text'][:400])
    print("=" * 60)

Total chunks in file: 2737


Sample 1 — chunk_2213 [table]
Breadcrumb : Bulb Replacement
Tokens     : 387
Characters : 1239
Text preview:
----------------------------------------
## Bulb Replacement

76TS90581

| No .                                                                                   | ITEM: Lights                   | WATT- AGE   | BULB No.   |
|----------------------------------------------------------------------------------------|--------------------------------|-------------|------------|
| (1)                                                             

Sample 2 — chunk_0403 [paragraph]
Breadcrumb : (Adjusting the date) (if equipped)
Tokens     : 82
Characters : 264
Text preview:
----------------------------------------
## (Adjusting the date) (if equipped)

- \u00b7 Adjust the date by selecting \"Clock setting\" in \"Setting mode\". Then select \"Adjust date\".
- \u00b7 To adjust year, month and day, operate the indicator selector knob (3) in the same way as adjus